# Module 3 POC Training -- EfficientNet-B4 + Weibull Survival Head

Trains Module 3 (time-to-progression estimation) on Tianjin's baseline -> 2-year-follow-up
DR grades, resolved per eye (Organized_Data of Patients.xlsx grades DR per eye, not per
patient): an EfficientNet-B4 backbone (ImageNet-pretrained, independently trained from
Module 1's classifier -- no shared weights) with a Weibull survival head, using the LBS
stratum (low/medium/high, see `module1/compute_lbs.py`) as an auxiliary input.

**Run this after notebook 05** (it reuses Tianjin's `module1_outputs_tianjin.pt` cache and
`laterality_resolved.csv` -- Module 3 has no placeholder fallback for either).

Tianjin's fixed 2-year follow-up window means this can only support a "probability of
progression by 2 years" claim, not a full time-to-event curve at arbitrary horizons. Report
it as a population-level probabilistic estimate with the AUROC below, not a per-patient point
prediction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'main'  # <-- change if this Module 3 work isn't merged to main yet
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
TIANJIN_CACHE = os.path.join(DRIVE_DATA_DIR, 'module1_cache', 'module1_outputs_tianjin.pt')
assert os.path.isfile(TIANJIN_CACHE), (
    f"Not found: {TIANJIN_CACHE} -- run notebook 05 first (it builds this cache)."
)

In [ ]:
# Unzip Tianjin locally (same convention as notebooks 05/06).
import glob

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/retinal-dr-longitudinal.zip" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found -- see notebook 05 for troubleshooting.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)
print('Tianjin manifest found:', os.path.isfile(os.path.join(TIANJIN_DIR, 'corrected_manifest.csv')))

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl scikit-learn opencv-python-headless

In [ ]:
# Laterality resolution: Organized_Data of Patients.xlsx grades DR per eye (OS/OD), not per
# patient, and corrected_manifest.csv doesn't say which eye a row is -- tianjin_dataset.py /
# module3/dataset.py both require laterality_resolved.csv to exist in TIANJIN_DIR before they
# can run. Reuse a Drive-persisted copy if one already exists (resolving ~1,100 images takes a
# few minutes); otherwise compute it fresh here and persist it to Drive for next time.
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, 'module1_cache', 'laterality_resolved.csv')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    import shutil
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive: {LATERALITY_DRIVE_PATH}')
else:
    %cd {REPO_CODE_DIR}
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"
    # Read the printed same-eye disagreement rate above before trusting this for training --
    # see resolve_eye_laterality.py's module docstring (>5% disagreement means the heuristic
    # needs tuning).
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    import shutil
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive: {LATERALITY_DRIVE_PATH}')

In [ ]:
# Sanity-check the survival dataset builds correctly against the real download before
# committing to a full training run -- prints patient counts, event rate, and LBS stratum
# thresholds per grade.
%cd {REPO_CODE_DIR}/module3
!python dataset.py --dataset-dir "{TIANJIN_DIR}" --module1-cache "{TIANJIN_CACHE}"

In [ ]:
%cd {REPO_CODE_DIR}/module3
!python train_module3_poc.py \
    --tianjin-dir "{TIANJIN_DIR}" \
    --tianjin-module1-cache "{TIANJIN_CACHE}" \
    --num-epochs 10

In [ ]:
import json, os

rp = f'{REPO_CODE_DIR}/module3/module3_runs_poc/poc_results.json'
print(json.dumps(json.load(open(rp)), indent=2) if os.path.isfile(rp) else f'{rp} not found -- the training cell above did not finish.')

## Caveats

- Trained on Tianjin only -- FIRE and LongDRScreening have no visit-date metadata at all, so
  they can't feed a survival model regardless of architecture.
- Grading is resolved per eye, not per patient -- both eyes of a patient are kept as
  independent samples, split at the patient level so no patient's two eyes leak across
  train/val (see `module3/dataset.py::patient_level_split`). Rows whose eye couldn't be
  confidently resolved fall back to the patient-level "worse eye" summary grade for that row
  only -- check the printed eye-specific-vs-fallback counts before trusting the usable-N
  number below.
- Stage 0 (No DR) baselines are in scope -- a real data-volume win (hundreds of Tianjin
  patients start at Stage 0), but their Lesion Burden Scores cluster near zero with little
  spread, so the low/medium/high LBS stratification may not discriminate much within Stage 0
  specifically.
- The model conditions on the baseline eye's own severity grade (a learned embedding alongside
  the LBS stratum), letting it distinguish "mild->moderate" risk from "severe->PDR" risk.
- Every Tianjin patient shares the same fixed 2-year follow-up interval. The reported AUROC
  and mean-predicted-vs-actual rate are for "probability of progression by 2 years"
  specifically -- the model has no training signal at any other time horizon.
- "Progression" = any upward ICDR stage move between baseline and the 2-year follow-up for
  that eye, not necessarily progression to sight-threatening disease specifically. The
  dataset's own "Progression (1=No;2=Yes)" column is logged as a cross-check (per-patient, not
  per-eye).
- POC scale: ImageNet-pretrained EfficientNet-B4 fine-tuned (not trained from scratch), small
  post-exclusion N, patient-level train/val split -- not a claim of a clinically validated
  survival model.